In [82]:
#keras sequence add
#pytorch class 수정 쉬움
#fashionmnist

#데이터 불러오기
from torchvision import datasets
from torchvision.transforms import ToTensor

In [83]:
#학습 데이터 불러오기
train_data=datasets.FashionMNIST(
    root="../Data/",
    train=True,
    download=True,
    transform=ToTensor()
)

In [84]:
#테스트 데이터 불러오기
test_data=datasets.FashionMNIST(
    root="../Data/",
    train=False,
    download=True,
    transform=ToTensor()
)#여기서 바꿨음

In [85]:
#받은 데이터 확인
print(train_data.data.shape)
print(train_data.targets.shape)

print(test_data.data.shape)
print(test_data.targets.shape)

torch.Size([60000, 28, 28])
torch.Size([60000])
torch.Size([10000, 28, 28])
torch.Size([10000])


In [86]:
#data와 target 분류
train_input=train_data.data
train_target=train_data.targets

test_input=test_data.data
test_target=test_data.targets

In [87]:
#target 확인
train_target[:10]

tensor([9, 0, 0, 3, 0, 2, 7, 2, 5, 5])

In [88]:
import numpy as np
print(np.unique(train_target,return_counts=True))

(array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]), array([6000, 6000, 6000, 6000, 6000, 6000, 6000, 6000, 6000, 6000]))


In [89]:
#데이터 표준화 및 2차원 행렬
train_scaled=(train_input/255.0).reshape(-1,28*28)
test_scaled=(test_input/255.0).reshape(-1,28*28)

print(train_scaled.shape)
print(test_scaled.shape)

torch.Size([60000, 784])
torch.Size([10000, 784])


----
#모델 만들기

In [90]:
import torch
import torch.nn as nn#뉴멀 네트워크?
import torch.optim as optim#optimize
from torch.utils.data import TensorDataset,DataLoader#셔플링

In [91]:
#Train과 Valid
from sklearn.model_selection import train_test_split
train_scaled, val_scaled,train_target,val_target=train_test_split(
    train_scaled,train_target,test_size=0.2,random_state=42
)

In [92]:
#dataset과 dataloader(그대로 사용하지 못함) 생성
batch_size=32#mini batch#묶어서 async로 돌린다.
train_dataset=TensorDataset(train_scaled,train_target)
train_loader=DataLoader(train_dataset,batch_size=batch_size,shuffle=True)#머신러닝 crossvalidate
val_dataset=TensorDataset(val_scaled,val_target)
val_loader=DataLoader(val_dataset,batch_size=batch_size)#검증 셔플 필요없음 학습시에만 에포크, 셔플

#모델 정의

In [93]:
# from turtle import forward

# class SimpleModel(nn.Module):#상속(initialize)생성시
#     def __init__(self):#설정을 해줘야함
#         super(SimpleModel, self).__init__()
#         self.dense=nn.Linear(28*28,10)#feature,target #인공신경망 히든레이어없음,입력층 출력층만 있음
#         self.softmax=nn.Softmax(dim=1)#정의 출력층/차원:1개
        
#     def forward(self,x):
#         x=self.dense(x)
#         return self.softmax(x)

In [94]:
class SimpleModel(nn.Module):
    def __init__(self):
        super(SimpleModel, self).__init__()
        self.dense = nn.Linear(28*28, 10)
        self.softmax = nn.Softmax(dim=1)
       
    def forward(self, x):
        # x = x.view(x.size(0), -1)
        x = self.dense(x)
        return self.softmax(x)

In [95]:
#모델 인스턴스
model=SimpleModel()

In [96]:
#손실함수와 옵티마이저
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

---
#모델 훈련

In [97]:
#학습함수(한번 만들고 복사):순서중요
def train(model,train_loader,criterion,optimizer,device):
    model.train()#dropout 포함됨
    # total_loss=0
    for inputs,targets in train_loader:
        inputs, targets =inputs.to(device), targets.to(device)#gpu에 옮겨놔야 계산을 해줌<<<<<<<<
        optimizer.zero_grad()#기울기 0:gradient 초기화
        outputs=model(inputs)#정방향
        loss=criterion(outputs,targets)#손실함수
        loss.backward()#역전파
        optimizer.step()#파라미터/가중치 바꿔줌
    return loss.item()

In [98]:
#평가함수
def evaluate(model, val_loader,criterion,device):
    model.eval()#llm:정방향:기울기?미분 바꿈
    total_loss=0#전체 손실 합계
    correct=0#정확하게 예측한 샘플수
    total=0#전체 샘플수
    with torch.no_grad():
        #with file close 안해도됨
        for inputs, targets in val_loader:
            inputs,targets= inputs.to(device),targets.to(device)
            outputs=model(inputs)
            loss=criterion(outputs,targets)
            total_loss+=loss.item()
            _, predicted= outputs.max(1)#선형회귀
            total+=targets.size(0)
            correct+=predicted.eq(targets).sum().item()

    return total_loss/len(val_loader),correct/total #평균손실, 정확도        


---
#학습 및 평가

In [99]:
device=torch.device("mps" if torch.backends.mps.is_available() else 'cpu')
print(device)
model.to(device)#summary

mps


SimpleModel(
  (dense): Linear(in_features=784, out_features=10, bias=True)
  (softmax): Softmax(dim=1)
)

In [100]:
# device=torch.device("cuda" if torch.cuda.is_available() else 'cpu')
# print(device)

In [101]:
#epochs for문
num_epochs=100
for epoch in range(num_epochs):
    train_loss=train(model, train_loader,criterion,optimizer,device)#loss만 return
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss:{train_loss:.4f}")

Epoch [1/100], Loss:1.7124
Epoch [2/100], Loss:1.5657
Epoch [3/100], Loss:1.6295
Epoch [4/100], Loss:1.5301
Epoch [5/100], Loss:1.6236
Epoch [6/100], Loss:1.5836
Epoch [7/100], Loss:1.4972
Epoch [8/100], Loss:1.5757
Epoch [9/100], Loss:1.6283
Epoch [10/100], Loss:1.5603
Epoch [11/100], Loss:1.6361
Epoch [12/100], Loss:1.5858
Epoch [13/100], Loss:1.5617
Epoch [14/100], Loss:1.5787
Epoch [15/100], Loss:1.5563
Epoch [16/100], Loss:1.5709
Epoch [17/100], Loss:1.6568
Epoch [18/100], Loss:1.6292
Epoch [19/100], Loss:1.6657
Epoch [20/100], Loss:1.6017
Epoch [21/100], Loss:1.7651
Epoch [22/100], Loss:1.6474
Epoch [23/100], Loss:1.6181
Epoch [24/100], Loss:1.5221
Epoch [25/100], Loss:1.5036
Epoch [26/100], Loss:1.6715
Epoch [27/100], Loss:1.6311
Epoch [28/100], Loss:1.6432
Epoch [29/100], Loss:1.5787
Epoch [30/100], Loss:1.5172
Epoch [31/100], Loss:1.5366
Epoch [32/100], Loss:1.5705
Epoch [33/100], Loss:1.5765
Epoch [34/100], Loss:1.6695
Epoch [35/100], Loss:1.5947
Epoch [36/100], Loss:1.5402
E

In [102]:
#훈련평가
# from sympy import evaluate

train_loss, train_accuracy = evaluate(model, train_loader, criterion, device)
print(f"Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.4f}")

#검증평가
# from sympy import evaluate


val_loss,val_accuracy=evaluate(model,val_loader,criterion,device)
print(f"Loss:{val_loss},Accuracy:{val_accuracy}")

#일반화 평가
test_dataset=TensorDataset(test_scaled, test_target)
test_loader=DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

#평가하기
test_loss,test_accuracy=evaluate(model,test_loader,criterion,device)
print(f"Loss:{test_loss},Accuracy:{test_accuracy}")

Train Loss: 1.5711, Train Accuracy: 0.8962
Loss:1.6011984113057454,Accuracy:0.8615833333333334
Loss:1.6118498263648524,Accuracy:0.8509
